# 05 — Comparación de modelos

Juntamos todo: los baselines (01) y el mejor modelo de cada familia según su
búsqueda de hiperparámetros (02–04), entrenados sobre el mismo train y evaluados
sobre el mismo test (≥2021). Los hiperparámetros de abajo son los ganadores
reportados en los notebooks 02, 03 y 04.

**Métrica principal: RMSE (kg/ha).** Reportamos también MAE, R² y MAPE.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))          # componente_b/ (datos, evaluacion)
warnings.filterwarnings('ignore')                  # silenciar ConvergenceWarning de sklearn

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

import datos, evaluacion as ev
from modelos import LinearRegressor, XGBoostRegressor, NeuralNetRegressor

# Cultivo del estudio (cambiar a 'maiz' para reproducir con maíz).
CULTIVO = 'soja'
ds = datos.prepare(CULTIVO)
print(f'{CULTIVO}: {len(ds.feature_cols)} features | '
      f'train {ds.X_train.shape[0]} filas (≤{datos.TRAIN_END}) | '
      f'test {ds.X_test.shape[0]} filas (≥{datos.TEST_START})')


## Mejores configuraciones (de los notebooks 02–04)

In [ ]:
best_linear = {'penalty': 'lasso', 'alpha': 10.0}
best_xgb = {'max_depth': 3, 'learning_rate': 0.03, 'n_estimators': 400,
            'subsample': 0.7, 'colsample_bytree': 0.8, 'min_child_weight': 5,
            'reg_lambda': 1.0, 'reg_alpha': 0.0, 'gamma': 1.0}
best_nn = {'hidden_dims': (64, 32), 'dropout': 0.5, 'weight_decay': 1e-3, 'lr': 3e-3}

## Entrenamiento y evaluación en test

In [ ]:
preds = {}
preds['media global']  = ev.pred_media(ds)
preds['media x depto'] = ev.pred_media_depto(ds)
preds['Lineal']  = LinearRegressor(**best_linear).fit(ds.X_train, ds.y_train).predict(ds.X_test)
preds['XGBoost'] = XGBoostRegressor(**best_xgb, random_state=42).fit(ds.X_train, ds.y_train).predict(ds.X_test)
preds['Red neuronal'] = NeuralNetRegressor(**best_nn, max_epochs=250, patience=30,
                                           random_state=42).fit(ds.X_train, ds.y_train).predict(ds.X_test)

filas = [ev.evaluar(k, v, ds) for k, v in preds.items()]
tabla = ev.tabla(filas)     # presentación estándar (RMSE/R²/MAE), la reusan los plots
tabla

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ev.plot_comparativa(tabla, metrica='rmse', ax=axes[0])
ev.plot_comparativa(tabla, metrica='r2', ax=axes[1])
plt.tight_layout(); plt.show()

## Predicho vs. real de los tres modelos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (nombre, color) in zip(axes, [('Lineal', ev.C_LINEAR),
                                      ('XGBoost', ev.C_XGB),
                                      ('Red neuronal', ev.C_NN)]):
    ev.plot_pred_vs_real(ds.y_test, preds[nombre], nombre, color=color, ax=ax)
plt.tight_layout(); plt.show()

## Conclusión

- Todos los modelos con clima **superan a la media global**; el listón real es la
  **media por departamento** (estructura espacial pura).
- **XGBoost** es el mejor: las interacciones no lineales entre clima, espacio y
  tendencia le dan la ventaja sobre el lineal, con la red neuronal cerca.
- El margen sobre el baseline por depto cuantifica cuánto aporta el **clima del año**
  por encima de "cada depto rinde lo de siempre" — que es, en el fondo, lo difícil
  de predecir del rinde.

## ¿Por qué el R² es "bajo" (~0.25–0.30)?

No es (solo) que falte tunear: hay un **techo estructural** en predecir rinde de
clima a este nivel de agregación.

1. **La varianza reducible es chica.** El grueso del rinde es *espacial* (qué depto)
   y *tendencia* (qué década) — cosas que los baselines ya capturan. Lo que queda por
   explicar con el **clima del año** es una porción menor y ruidosa, así que aunque
   el modelo la capture bien, el R² total sube poco.
2. **Agregación depto-campaña.** Cada fila promedia miles de lotes con siembras,
   cultivares, suelos y manejo distintos: la relación clima→rinde de lote se diluye.
3. **Variables no observadas.** Manejo (fertilización, fecha de siembra, genética),
   plagas/enfermedades, granizo, y ruido de reporte de MAGYP no están en las features.
   El Componente A ya mostró que ~2/3 de las anomalías de rinde **no tienen firma
   climática**.
4. **Extrapolación temporal.** El test (2021–2024) cae *fuera* del rango de train: la
   tendencia y el régimen climático se corren. Se ve en que la feature `year`
   extrapola (ayuda al lineal, la cortan los árboles) y en que el `es_anomalo` del
   VAE marca casi todo el test (el score sube por *distribution shift*, no por
   anomalía real).

**Sobre el latente del Componente A** (nbs 02–04): concatenar el **latente del VAE**
suele empeorar la regresión (y solo-latente es lo peor): ese latente resume el clima
*normalizado por depto*, así que tira la señal espacial y de tendencia que es justo la
que más predice. `es_anomalo` tampoco ayuda, dominado por el *distribution shift*.
Coherente con el techo del Componente A.

## ¿Qué podemos modificar para mejorar?

En orden aproximado de impacto esperado:

1. **Cambiar el target a algo aprendible.** En vez del rinde absoluto, predecir la
   **anomalía de rinde** (residuo sobre la media/tendencia por depto, p. ej. el
   `z_rinde` del Componente A). Saca la parte "fácil" (espacio+tendencia) y deja que
   el modelo se concentre en la señal climática — R² más honesto de lo que sí se
   puede predecir.
2. **Features agronómicas, no promedios mensuales crudos.** Índices en la **ventana
   crítica** por cultivo: balance hídrico acumulado, días de estrés térmico (Tmax>32
   en floración), rachas secas, grados-día. El Componente A ya tiene `add_agro_features`.
3. **NDVI como predictor directo, no como una feature más.** El NDVI de
   floración/llenado es un proxy casi directo del rinde; conviene usarlo con más peso
   (o un modelo aparte) en vez de mezclarlo entre 60+ columnas.
4. **Tratar la extrapolación temporal.** Detrendear el rinde antes de modelar, o usar
   validación que imite el gap train→test; para los árboles, no darles `year` crudo.
5. **Modelo residual / híbrido.** Baseline fuerte = media por depto (+ tendencia), y
   un modelo que aprenda **solo el residuo** con el clima. Suele ganarle a predecir
   el absoluto de una.
6. **Robustez al ruido de etiqueta.** Pérdida robusta (Huber/cuantil) por el ruido de
   reporte de MAGYP, y quizás modelar por región/cultivo por separado.

La limitación de fondo (agregación depto-campaña + variables no observadas) solo se
levanta con **datos más finos** (lote/píxel, manejo), que exceden este panel.

Para reproducir con maíz: cambiar `CULTIVO = 'maiz'` en la celda de setup y
re-ejecutar los notebooks.